# Sample v1 (per email)

## Totais gerais

In [ ]:
import pandas as pd
import re

pd.set_option('display.max_colwidth', None)

v1 = pd.read_csv('sample_review_v1_per_email.csv', dtype=str, keep_default_na=False)
usp = pd.read_csv('../v2/sample_review_usp_v2.csv', dtype=str, keep_default_na=False)

v1_overall = pd.DataFrame({
    'metric': ['total_emails', 'total_threads', 'yes_emails', 'no_emails'],
    'value': [
        len(v1),
        v1['thread_id'].nunique(),
        (v1['is_clone_refactoring'] == 'yes').sum(),
        (v1['is_clone_refactoring'] == 'no').sum(),
    ],
})
v1_overall

## Por lista (nível de email)

In [2]:
v1_by_list_email = (
    v1.groupby('list')['is_clone_refactoring']
    .value_counts()
    .unstack(fill_value=0)
    .reindex(columns=['yes', 'no'], fill_value=0)
)
v1_by_list_email

is_clone_refactoring,yes,no
list,,
amd,20,102
iio,7,45
linux-i2c,6,20


## Por lista (nível de thread)

In [3]:
v1_thread_label = v1.groupby(['list', 'thread_id'])['is_clone_refactoring'].apply(
    lambda s: 'yes' if 'yes' in set(s) else 'no'
).reset_index()

v1_by_list_thread = (
    v1_thread_label.groupby('list')['is_clone_refactoring']
    .value_counts()
    .unstack(fill_value=0)
    .reindex(columns=['yes', 'no'], fill_value=0)
)
v1_by_list_thread

is_clone_refactoring,yes,no
list,,
amd,15,47
iio,7,18
linux-i2c,6,7


# Sample USP (per thread)

## Totais gerais

In [4]:
usp_overall = pd.DataFrame({
    'metric': ['total_threads', 'yes_threads', 'no_threads'],
    'value': [
        len(usp),
        (usp['is_clone_refactoring'] == 'yes').sum(),
        (usp['is_clone_refactoring'] == 'no').sum(),
    ],
})
usp_overall

,metric,value
0,total_threads,112
1,yes_threads,36
2,no_threads,76


## Por lista

In [5]:
usp_by_list = (
    usp.groupby('list')['is_clone_refactoring']
    .value_counts()
    .unstack(fill_value=0)
    .reindex(columns=['yes', 'no'], fill_value=0)
)
usp_by_list

is_clone_refactoring,yes,no
list,,
amd,19,15
iio,17,61


# Precisão por termo (v1 + USP combinados, nível de thread)

In [6]:
v1_thread_content = (v1['subject'] + ' ' + v1['raw_body']).groupby(v1['thread_id']).apply(lambda s: ' '.join(s))
v1_thread_label_only = v1.groupby('thread_id')['is_clone_refactoring'].apply(
    lambda s: 'yes' if 'yes' in set(s) else 'no'
)
v1t = pd.DataFrame({'label': v1_thread_label_only, 'content': v1_thread_content}).reset_index(drop=True)

usp_t = usp[['is_clone_refactoring', 'thread_content']].rename(
    columns={'is_clone_refactoring': 'label', 'thread_content': 'content'}
).reset_index(drop=True)

combined = pd.concat([v1t, usp_t], ignore_index=True)

# content agora inclui subject+body (como o thread_content real do pré-filtro),
# não só raw_body — sem isso, 1 thread v1 'yes' cujo termo só aparece no
# subject ("reduce code duplication in tsl2591_persist_* functions") vira
# falso negativo artificial (ver PLANO_pre_filter_threads.md, seção de
# validação de recall).

TERMS = {
    'duplicat*': r'duplicat\w*',
    'dedup*': r'dedup\w*',
    'redundant*': r'redundant\w*',
    'repeated': r'repeated',
    'clon(e/ed/ing/es)': r'clon(?:e|ed|ing|es)?',
    'copy[-_]?past(e/ed/ing)': r'copy[-_\s]?past(?:e|ed|ing)',
}

# precision = yes_matched / threads_matched: fração das threads (v1 + usp, uma linha por thread)
# que contêm o termo e foram classificadas como yes
rows = []
for name, pat in TERMS.items():
    rgx = re.compile(r'\b(?:' + pat + r')\b', re.IGNORECASE)
    hit = combined['content'].apply(lambda t: bool(rgx.search(t or '')))
    sub = combined[hit]
    n_hit = len(sub)
    n_yes = (sub['label'] == 'yes').sum()
    precision = n_yes / n_hit if n_hit else float('nan')
    rows.append({'term': name, 'threads_matched': n_hit, 'yes_matched': n_yes, 'precision': round(precision, 2)})

term_precision = pd.DataFrame(rows).sort_values('precision', ascending=False).reset_index(drop=True)
term_precision

,term,threads_matched,yes_matched,precision
0,dedup*,19,17,0.89
1,copy[-_]?past(e/ed/ing),12,8,0.67
2,duplicat*,83,52,0.63
3,repeated,15,6,0.40
4,redundant*,53,13,0.25
5,clon(e/ed/ing/es),5,0,0.00


## Citações

In [7]:
def excerpt(text, needle, pad=150):
    m = re.search(re.escape(needle), text, re.IGNORECASE)
    if not m:
        return ''
    s = max(0, m.start() - pad)
    e = min(len(text), m.end() + pad)
    return text[s:e].replace('\n', ' ').strip()

examples = [
    {
        'false_positive': 'yes',
        'thread_id': 'a5f06933-3cbc-4b6d-99fc-23ee31b291ca@amd.com',
        'needle': 'declared twice',
    },
    {
        'false_positive': 'yes',
        'thread_id': 'aeXnujb9-aui4ArO@ashevche-desk.local',
        'needle': "'else' is redundant",
    },
    {
        'false_positive': 'no',
        'thread_id': '20260513003503.339418-2-guilherme.bozi@usp.br',
        'needle': 'switch statements spread',
    },
    {
        'false_positive': 'no',
        'thread_id': None,
        'needle': 'call common helper function to init sos ucode',
    },
]

rows = []
for ex in examples:
    if ex['thread_id'] is not None and ex['thread_id'] in set(usp['thread_id']):
        text = usp.loc[usp['thread_id'] == ex['thread_id'], 'thread_content'].iloc[0]
    else:
        pool = pd.concat([usp['thread_content'], v1['raw_body']], ignore_index=True)
        text = next((t for t in pool if ex['needle'].lower() in str(t).lower()), '')
    rows.append({
        'quote': excerpt(str(text), ex['needle']),
        'false_positive': ex['false_positive'],
    })

examples_df = pd.DataFrame(rows)
examples_df

,quote,false_positive
0,"splay: Remove duplicated function signature from dcn3.01 DCCG Email body: In the header file dc/dcn301/dcn301_dccg.h, the function dccg301_create is declared twice, so remove duplication. Signed-off-by: David Tadokoro <davidbtadokoro@usp.br> --- drivers/gpu/drm/amd/display/dc/dcn301/dcn301_dccg.h | 6 ------ 1",yes
1,"v_err(indio_dev->dev.parent, > \t\t\t\t""Failed to re-initialize device: %d\n"", ret); > -\t\t\tret = NOTIFY_BAD; > +\t\t\treturn NOTIFY_BAD; > \t\t} else { Now 'else' is redundant. > -\t\t\tret = NOTIFY_OK; > +\t\t\treturn NOTIFY_OK; > \t\t} > - > -\t\tgoto err_mutex; ... > \tstruct iqs621_als_private *iqs621_als = iio_priv(indio_dev",yes
2,"logic across the Xilinx AMS IIO driver. To address this, the series introduces a centralized table-driven mapping (alarm_map) that replaces multiple switch statements spread across the driver. This improves: - maintainability (single source of truth for mappings) - readability (removes repeated switch logic) - extensibil",no
3,"call common helper function to init sos ucode, instead of duplicate codes per ip version Signed-off-by: Hawking Zhang <Hawking.Zhang@amd.com> --- drivers/gpu/drm/amd/amdgpu/psp_v11_0.c | 56 ++-",no


Excluir `clone*` do regex final do pré-filtro faz 3 das 212 threads da amostra (v1+usp) deixarem de casar — todas rotuladas `no`, ou seja, nenhum verdadeiro positivo é perdido (recall permanece 100%). A busca por `clone*` também foi feita, sem gold label, em mais duas listas fora do gold (`openrisc` e `driver-core`, ver `PLANO_pre_filter_threads.md`) e também não trouxe nenhum verdadeiro positivo: 89 hits no total (43 em `openrisc`, 46 em `driver-core`), todos referentes a `clone()`/`sys_clone`/`CLONE_*` (syscall), `git clone`, ao alvo de storage `dm-clone`, ou à trait `Clone`/`.clone()` do Rust — nenhum sobre duplicação de código-fonte.

# Pré-filtro (regex) — threads antes x depois, por lista

In [ ]:
import glob
import os

import pyarrow.parquet as pq

BUILD_THREADS_DIR = '../../build_threads_output'
PRE_FILTER_DIR = '../../pre_filter/pre_filter_output'

def n_rows(path):
    return pq.ParquetFile(path).metadata.num_rows

pre_filter_rows = []
for path in sorted(glob.glob(f'{BUILD_THREADS_DIR}/list_data_*.parquet')):
    list_name = os.path.basename(path)[len('list_data_'):-len('.parquet')]
    before = n_rows(path)
    after = n_rows(f'{PRE_FILTER_DIR}/list_data_{list_name}.parquet')
    pre_filter_rows.append({'list': list_name, 'threads_before': before, 'threads_after': after})

pre_filter_counts = pd.DataFrame(pre_filter_rows)
pre_filter_counts['pct_after'] = (pre_filter_counts['threads_after'] / pre_filter_counts['threads_before'] * 100).round(2)

top10 = pre_filter_counts.sort_values('threads_before', ascending=False).head(10)

total_before = pre_filter_counts['threads_before'].sum()
total_after = pre_filter_counts['threads_after'].sum()
total_row = pd.DataFrame([{
    'list': f'TOTAL ({len(pre_filter_counts)} listas)',
    'threads_before': total_before,
    'threads_after': total_after,
    'pct_after': round(total_after / total_before * 100, 2),
}])

pre_filter_top10 = pd.concat([top10, total_row], ignore_index=True)
pre_filter_top10['pct_after'] = pre_filter_top10['pct_after'].astype(str) + '%'
pre_filter_top10

# Teste classify_threads.py (amostra completa, pós pré-filtro novo)

In [9]:
import json

# 81/100 threads da amostra gold sobrevivem ao pré-filtro atual (o antigo
# arquivo era só a lista amd, 62 threads, sem passar pelo pré-filtro).
postfilter = pd.read_parquet('sample_review_v1_postfilter_classified.parquet')

def predicted_label(findings_json):
    cats = {f['category'] for f in json.loads(findings_json)}
    if cats == {'not_duplication'}:
        return 'no'
    if cats == {'inconclusive'}:
        return 'inconclusive'
    return 'yes'

postfilter['predicted'] = postfilter['llm_findings'].apply(predicted_label)
len(postfilter)

81

## Métricas (classe yes, threads decididas)

In [10]:
decided = postfilter[postfilter['predicted'] != 'inconclusive']

tp = ((decided['is_clone_refactoring'] == 'yes') & (decided['predicted'] == 'yes')).sum()
fp = ((decided['is_clone_refactoring'] == 'no') & (decided['predicted'] == 'yes')).sum()
fn = ((decided['is_clone_refactoring'] == 'yes') & (decided['predicted'] == 'no')).sum()
tn = ((decided['is_clone_refactoring'] == 'no') & (decided['predicted'] == 'no')).sum()

metrics = pd.DataFrame({
    'metric': ['accuracy', 'precision_yes', 'recall_yes', 'f1_yes', 'n_decided', 'n_inconclusive'],
    'value': [
        round((tp + tn) / len(decided), 3),
        round(tp / (tp + fp), 3),
        round(tp / (tp + fn), 3),
        round(2 * (tp / (tp + fp)) * (tp / (tp + fn)) / ((tp / (tp + fp)) + (tp / (tp + fn))), 3),
        len(decided),
        len(postfilter) - len(decided),
    ],
})
metrics

,metric,value
0,accuracy,0.765
1,precision_yes,0.615
2,recall_yes,0.960
3,f1_yes,0.750
4,n_decided,68.000
5,n_inconclusive,13.000
